# Part 1 — Acquisition & Storage: Data Acquisition and Source Verification

BFT65B · Student ID 11230945

Acquire every Financials quarter from 1984Q1 to 2026Q2, plus both complete reference tables. Preserve source fields and create a verified Bronze run. No labels or economic corrections are performed here.

## 1. Repository and pinned runtime
Run on a CPU runtime. The package installation and research code use separate Python processes.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/NguyenDuong0508/11230945.git"
REPO = Path("/content/11230945")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    actual = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip()
    if actual != REPO_URL:
        raise RuntimeError(f"Unexpected repository: {actual}")
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)
# The research commands run in fresh Python processes to avoid stale imports.
def command(*args):
    subprocess.run([sys.executable, str(REPO / "run_part1.py"), *map(str, args)], cwd=REPO, check=True)


## 2. Mandatory offline validation
The tests cover source contracts, pagination, checkpoint recovery, CSV schema restoration, Parquet equality and partition pruning. Synthetic fixtures do not call the live API. Continue only after every test passes without skips.

In [ ]:
import tempfile
import xml.etree.ElementTree as ET

TESTS_PASSED = False
with tempfile.TemporaryDirectory() as tmp:
    result_path = Path(tmp) / "tests.xml"
    env = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
    # Stream the complete failure traceback instead of truncating captured output.
    subprocess.run([sys.executable, "-m", "pytest", "-q", "--tb=short",
                    "--show-capture=no", f"--junitxml={result_path}"],
                   cwd=REPO, env=env, check=True)
    suites = ET.parse(result_path).findall(".//testsuite")
    total = sum(int(s.get("tests", 0)) for s in suites)
    skipped = sum(int(s.get("skipped", 0)) for s in suites)
    if total == 0 or skipped:
        raise RuntimeError(f"Incomplete tests: total={total}, skipped={skipped}")
    TESTS_PASSED = True
    print(f"Mandatory offline validation: PASS ({total} tests, zero skips)")


## 3. Persistent workspace
Keep the source data in Google Drive, outside GitHub. Use this workspace for all subsequent parts.

In [ ]:
from google.colab import drive
import json

drive.mount("/content/drive")
WORKSPACE = Path("/content/drive/MyDrive/11230945")
WORKSPACE.mkdir(parents=True, exist_ok=True)
print("Workspace:", WORKSPACE)


## 4. First-quarter acquisition check
The initial command acquires at most one new financial quarter. It saves a checkpoint, not a complete source. If the workspace already has a sealed source, the command verifies it instead. This is also a live API field and transport check for the first quarter.

In [ ]:
assert TESTS_PASSED, "Run the full offline test suite first"
command("acquire", "--workspace", WORKSPACE, "--max-new-quarters", 1)


## 5. Full acquisition
Resume the pending source and retrieve all remaining quarters plus Failures and Institutions without outcome filters. Every completed artifact is committed to the manifest. If the runtime disconnects, rerun setup, tests, mount and this cell; completed valid checkpoints are reused. Do not start simultaneous acquisitions against the workspace.

In [ ]:
assert TESTS_PASSED
command("acquire", "--workspace", WORKSPACE)


## 6. Independent verification
Reopen the sealed artifacts and validate the calendar, API totals, physical files, keys, schema and checksums. The active pointer refers to the exact verified manifest.

In [ ]:
assert TESTS_PASSED
command("verify", "--workspace", WORKSPACE)
assert not (WORKSPACE / "pending_source.json").exists()
source_pointer = json.loads((WORKSPACE / "active_source.json").read_text())
SOURCE = WORKSPACE / source_pointer["path"]
source_report = json.loads((SOURCE / "verification.json").read_text())
assert source_report["status"] == "PASS" and source_report["quarter_count"] == 170
assert source_report["manifest_sha256"] == source_pointer["manifest_sha256"]
print("Acquisition accepted:", SOURCE.name)
print("Rows:", source_report["row_counts"])
print("Distinct financial CERTs:", source_report["distinct_financial_certs"])
print("Missing failure CERTs:", source_report["failure_null_cert"])
print("RESTYPE distribution (not model labels):", source_report["failure_restype_counts"])
print("Manifest SHA-256:", source_pointer["manifest_sha256"])


## 7. Quarterly coverage evidence
Compare retrieved rows with the API total for every quarter, not only the combined row count. Display a compact coverage summary.

In [ ]:
from IPython.display import display, Markdown
import html

quarter_stats = source_report["quarter_statistics"]
assert all(q["rows"] == q["api_total"] for q in quarter_stats)
# Small textual summary; no bank-level source rows are displayed.
first, last = quarter_stats[0], quarter_stats[-1]
for q in (first, last):
    print(q["quarter"], f"{q['rows']:,} rows", f"{q['bytes']/2**20:.2f} MiB")
print("Quarters with matching API totals:", len(quarter_stats))
print("Financial partition size (MiB):", round(sum(q["bytes"] for q in quarter_stats)/2**20, 3))


## 8. Handoff
Notebook 02 resolves `active_source.json`; do not copy a run ID. Part 2 will use the same immutable source and event fields. Save this executed notebook with its summary outputs, but do not commit the workspace or source data.